# Big Data Platforms — Lecture 7 Notes

Course: DATA140031 (MOOC, 3 ECTS) + DATA140032 (MOOC EXAM, 2 ECTS)
Lecturer: Keijo Heljanko, Department of Computer Science, University of Helsinki
Date: 22.9.2026

Lecture 6 was about *coordinating* distributed state — CAP, consensus, ZooKeeper.
Lecture 7 zooms into the databases that actually *store* that state at cloud
scale: the whole family of systems usually lumped together as "NoSQL." It works
through the standard taxonomy (key-value, document, extensible record, scalable
relational), the eventual-consistency/BASE idea that underlies a lot of them,
both sides of the "is SQL obsolete?" argument, and Google Spanner as the
counterexample that breaks the usual CAP-driven trade-offs. The back half is a
genuinely self-contained deep dive into **Bloom filters** — probably the most
important single data structure in this course for anyone who ends up building
systems rather than just using them — worked all the way from the definition
through a fully traced example to the real formula used to size one in
production. That worked example is reproduced as running code below, since
tracing it by hand once and then checking it in code is the best way to actually
trust the formula.


## 1. NoSQL databases, a.k.a. cloud datastores

Two names circle the same idea. **"Cloud datastore"** is used because these
systems were built for the cloud and often don't support every feature of a
traditional relational database (RDBMS). **"NoSQL"** stuck because the earliest
systems in this family genuinely didn't support SQL at all — though that's
becoming a less accurate label by the year, since SQL-like query support has
been creeping back into many cloud datastores. The lecture treats "NoSQL" as a
convenient but increasingly imprecise umbrella term rather than a precise
technical category.

These systems can be grouped along several different characteristics, and the
standard reference for comparing them is Rick Cattell's survey: *Scalable SQL
and NoSQL Data Stores*, SIGMOD Record 39(4), December 2010.
<https://doi.org/10.1145/1978915.1978919>


## 2. Common features across scalable cloud datastores

Cattell's paper identifies a set of features that recur across many of these
new datastores — **no single system has all of them**, but most have several:

- The ability to **horizontally scale** throughput for simple operations
  across many servers.
- Automatic **replication** and **partitioning/sharding** of data across many
  servers.
- A **simple call-level interface or protocol**, rather than a full SQL query
  language.
- A **weaker concurrency model** than the ACID transactions typical relational
  databases offer.
- Efficient use of **distributed indexes and RAM** for data storage.
- The ability to **dynamically add new attributes** to data records — no fixed,
  upfront schema.

Notice the shape of this list: almost every item is something a traditional
RDBMS deliberately does the opposite of, in the name of correctness and
generality. That's really the whole story of this lecture's first half — each
NoSQL family picks a different subset of these trade-offs, in exchange for
scaling further than a single-server RDBMS comfortably can.


## 3. The four-way grouping of data stores

### Key-value stores
- Examples: **Redis**, **Riak**, **Apache Cassandra** (partially),
  **Scalaris**.
- A unique primary key retrieves a data item — usually just a binary blob,
  though some systems support more structure than that.
- Commonly built on peer-to-peer techniques — a **distributed hash table
  (DHT)** and **consistent hashing** — to shard data and let servers be added
  or removed elastically without a disruptive full re-shard.
- Some systems are purely RAM-based (essentially a distributed memcached
  replacement); others persist to disk and can genuinely replace a persistent
  database.
- Most are **AP** systems (recall lecture 6's CAP framing), though a few — like
  Scalaris — support local, row-level transactions.
- **Cassandra resists easy categorization**: it uses DHT-style sharding, is an
  AP system by design, yet has a notably rich data model that goes well beyond
  a plain key-value store.

### Document stores
- Examples: **Amazon SimpleDB**, **CouchDB**, **MongoDB**.
- Built for storing structured documents — think XML, or more commonly JSON.
- Typically don't support transactions or full ACID semantics.
- Usually allow flexible indexing across many different document fields.
- The design priority here is explicitly **programmer productivity**, not
  squeezing out the absolute maximum scalability — a genuinely different goal
  from the other three families.

### Extensible record stores ("BigTable clones")
- Examples: **Google BigTable**, **Google Megastore**, **Apache HBase**,
  **Hypertable**, **Cassandra** (again — it straddles categories).
- Google's original BigTable paper introduced a database design approach that
  every system in this family has essentially emulated.
- BigTable's design is explicitly **write-optimized**: it trades away some read
  performance specifically to gain more write performance.
- Other defining traits: a **CP** design (consistent and partition tolerant,
  in Brewer's sense from lecture 6), automatic sharding keyed on the primary
  key, and a flexible data model.

### Scalable relational systems (a.k.a. distributed databases)
- Examples: **MySQL Cluster**, **VoltDB**, **Oracle Real Application Clusters
  (RAC)**.
- Data is sharded across a number of database servers, usually with automatic
  replication.
- Usually genuinely SQL-accessible, with **full ACID transaction support** —
  the closest of the four families to "an RDBMS, just distributed."
- The catch, for scalability: applications need to avoid joins or global
  transactions that span multiple database servers, since those are exactly
  the operations that don't scale horizontally.
- As of this lecture, scalability to very large deployments (100+ database
  servers) hadn't been demonstrated for this family — though there's no
  fundamental reason it couldn't be, just that nobody had proven it out at that
  scale yet.


## 4. Eventual consistency and BASE

This is a direct continuation of the AP side of lecture 6's CAP discussion,
given its own name and acronym.

The term **"eventually consistent"** was popularized by the authors of
**Amazon Dynamo** — an AP system by design, choosing availability and partition
tolerance over strict consistency. **BASE** — **B**asically **A**vailable,
**S**oft state, **E**ventually consistent — is essentially a synonym for the
same underlying approach, deliberately coined as a contrasting acronym to ACID.

The core idea: these datastores explicitly **value availability over strict
data consistency**. To make that livable in practice, they often provide
built-in support for automatically resolving at least some of the resulting
inconsistencies — version numbering being one common mechanism, and **CRDTs**
(Commutative Replicated Data Types — data structures specifically designed so
that merging two divergent copies always has one unambiguous, correct answer)
being a more sophisticated one. Example systems: Amazon Dynamo, Riak, Apache
Cassandra.


## 5. The RDBMS debate — two honest sides

Rather than declaring a winner, the lecture lays out both sides of the
"is SQL/RDBMS obsolete?" argument as it stood at the time — this is worth
reading as a genuine debate rather than a settled question.

### The RDBMS supporter's view
- A properly-used RDBMS can do everything scalable cloud datastores can, with
  comparable scalability.
- SQL is a convenient, expressive, **declarative** query language — you say
  what you want, not how to compute it.
- RDBMSs carry **30 years of engineering experience**, and are extremely
  well-tuned as a result.
- There's a large ecosystem of **specialized RDBMS variants** for different
  application domains — interactive vs. analytics workloads, RAM-resident vs.
  disk-based data, and so on.
- Standardizing around relational schemas and SQL pays off in **training and
  design reuse**: the same skill set transfers cleanly across different RDBMS
  products.

### The RDBMS opponent's view
- RDBMS vendors haven't demonstrated scalability matching the newer,
  purpose-built cloud datastores (BigTable being the headline example).
- A simple **primary-key lookup** is far easier to reason about than general
  SQL, giving a much gentler learning curve.
- RDBMSs **force a schema** on applications, often unnecessarily.
- SQL makes it dangerously **easy to write an expensive query by accident** —
  a complicated join across many tables and several servers can be typed in
  a single innocent-looking line. Strip that expressive power out of the query
  language, and the real cost of a query becomes obvious to the programmer up
  front, rather than hiding behind a convenient syntax.
- After 30 years dominating the database market, RDBMS systems have arguably
  become **dinosaurs** — built on hardware assumptions (limited RAM, expensive
  disk seeks, single powerful servers) that are increasingly obsolete.

Both views are presented as legitimate — the point isn't to pick a side, it's
to understand that the "NoSQL vs. SQL" argument is really an argument about
which trade-offs matter more for a given workload, not a simple technology
upgrade in either direction.


## 6. Where this was heading (future speculation, as of the lecture)

A few forward-looking observations the lecture offers, framed explicitly as
speculation rather than settled fact:

- **Long-running, globally serializable transactions are genuinely hard to
  implement efficiently**, and for scalability's sake should generally be
  avoided altogether. The lecture flags **Google Percolator** and **Google
  Cloud Spanner** as notable counterexamples — systems that took on exactly
  this hard problem deliberately, rather than avoiding it (Spanner gets its
  own section below).
- The demands of low latency and high availability make **AP** solutions
  attractive, but they're genuinely **harder for programmers to use** than CP
  systems, precisely because AP requires application-specific logic to recover
  from data inconsistencies — recall lecture 6's point that there's no generic
  algorithm for this.
- The era of "one size fits all," where a single RDBMS was the default answer
  to every datastore problem, **has passed** — scalable cloud datastores are
  here to stay as a genuinely distinct category, not a passing fad.
- That said, **ACID transactions remain necessary** for domains like financial
  transactions, where traditional RDBMS technology will likely remain
  dominant — this isn't a story of NoSQL replacing RDBMS everywhere, just
  displacing it from the workloads where its guarantees weren't actually
  needed.
- Many of the newer systems, at the time, **weren't yet proven in production**
  at real scale.
- The expectation was that the field would eventually **consolidate** to a
  smaller number of well-understood datastores, once the current burst of
  "design space exploration" settled down.


## 7. NewSQL — Google Cloud Spanner as the exception that proves the rule

Everything in the CAP discussion (lecture 6) and the AP-is-hard-to-program
discussion above points toward a seemingly unavoidable trade-off: **you can't
have both strict global consistency and good availability under partition.**
**Google Cloud Spanner** — the public release of Google's internal Spanner
database — is interesting precisely because it pushes back against that
trade-off harder than almost anything else in this lecture, using a genuinely
clever trick.

What Spanner claims to deliver:
- Full **SQL support**.
- **ACID transactions that can span multiple servers globally** — not just
  within one datacenter.
- A **fully consistent database with a single global image**, worldwide.

### How it actually pulls this off

- Replication is built on **Paxos** (lecture 6's consensus algorithm), giving
  it the same fault-tolerance foundation as ZooKeeper and Chubby.
- Transaction commit **priority/ordering** is determined using **physical
  timestamps**, rather than purely logical ones.
- This only works because Spanner does something almost no other distributed
  system attempts: it keeps **all database server clocks synchronized
  extremely tightly**, using a combination of **atomic clocks and GPS-based
  time synchronization** — hardware-level infrastructure most systems simply
  don't have access to.
- Because clock synchronization is never perfect, every transaction has to
  **wait for twice the maximum possible clock drift** while still holding its
  write locks, before it's allowed to commit — this deliberate wait is exactly
  what guarantees transactions across different servers get a globally
  consistent ordering, at the direct cost of added latency on every write.

Spanner essentially buys its way partway out of the CAP trade-off by spending
real money on synchronized physical clocks — a resource most organizations
running distributed databases don't have. Open-source systems inspired by the
same approach have since emerged, including **CockroachDB** and **Apache
Kudu**, though without Google's atomic-clock infrastructure, they typically
approximate Spanner's guarantees rather than matching them exactly.


## 8. Why probabilistic data structures matter here

This is the pivot into the second half of the lecture. Recall from lecture 5:
**RAM is dramatically more expensive per byte than hard disk or SSD**, and also
dramatically faster. Given that RAM is the scarce, valuable resource, using it
as efficiently as possible matters a great deal for any system trying to scale.

**Probabilistic data structures** trade a small, tunable amount of inaccuracy
for a dramatic reduction in memory footprint compared to an exact data
structure holding the same information. **Bloom filters** are the canonical
example, and they're heavily used inside real NoSQL databases specifically to
cut down on unnecessary disk I/O by keeping a small, RAM-resident probabilistic
index. The same idea shows up well beyond databases too — in data analytics,
Bloom filters and their relatives are routinely used to do fast approximate
computations over huge datasets.


## 9. Motivating problem — caching on second access

**The setup:** designing a caching layer for a web server. A stream of web page
URLs arrives, one per access. The goal: identify which URLs are being accessed
**more than once**, and cache only those — caching something accessed only once
risks evicting genuinely popular content to make room for a one-off hit, which
defeats the whole point of a cache. A small rate of **false positives** is
explicitly tolerable here: occasionally caching something on its first access
is a perfectly fine trade if it saves enough memory and speed to be worth it.

### Three candidate designs

1. **A hash table of every URL seen so far.** Simple and exact — cache a URL
   the moment it's seen a second time. The problem: this can require a *lot*
   of memory, since it has to store every distinct URL string ever seen.
2. **An external database of every URL seen so far.** Scales past what fits in
   main memory, but now every single cache-eligibility check needs a database
   query — potentially far too slow for something sitting in the hot path of
   every web request.
3. **A Bloom filter recording the set of URLs seen so far.** The design this
   lecture actually develops — dramatically less memory than option 1, and no
   external query round-trip like option 2.


## 10. Bloom filters — the definition

A Bloom filter is a highly memory-efficient **probabilistic data structure for
representing a set**.

**Structure:** a bit array *B* of *m* bits, *B[0], B[1], …, B[m−1]*, all
initialized to zero.

**Insertion of an item *d*:** compute *k* independent hash functions
*h₀(d), h₁(d), …, h_{k−1}(d)*, each mapping into the range *0 ≤ hᵢ(d) ≤ m−1*.
Then set the bit-array position at every one of those *k* hashed indices to 1:
*B[h₀(d)] = 1; B[h₁(d)] = 1; … B[h_{k−1}(d)] = 1*.

**Lookup of an item *d*:** compute the same *k* hash functions again. If
**every one** of those *k* positions in *B* currently holds a 1, the filter
answers **"the item is potentially in the set."** This can happen even for an
item that was *never* inserted — purely from hash collisions with other
inserted items — and that outcome is called a **false positive**. If **any**
of the *k* positions holds a 0, the filter answers **"the item is definitely
not in the set"** — and this answer is always correct, with zero possibility of
a false negative, since inserting an item can only ever set bits to 1, never
clear them back to 0.


## 11. Running example — tracing it by hand, then checking it in code

The lecture works a small, fully-traced example: an 8-bit Bloom filter
(*m = 8* — tiny, purely for demonstration; real filters run to megabytes or
more) with *k = 2* independent hash functions, *h₀* and *h₁*. Below, the exact
same hash outputs the lecture uses are hard-coded, so the trace matches the
slides bit for bit.


In [1]:
class TracedBloomFilter:
    """An 8-bit, k=2 Bloom filter with the *exact* hash outputs used in the
    lecture's worked example, so this trace matches the slides bit-for-bit.
    A real Bloom filter would use genuine hash functions -- see the
    general-purpose version further down."""

    # Hard-coded (item -> (h0, h1)) pairs, exactly as given in the lecture
    HASHES = {
        "http://www.cnn.com/":      (2, 5),
        "http://www.helsinki.fi/":  (5, 0),
        "http://www.yle.fi/":       (0, 2),
    }

    def __init__(self, m=8):
        self.m = m
        self.B = [0] * m

    def _hashes(self, item):
        return self.HASHES[item]

    def lookup(self, item):
        i0, i1 = self._hashes(item)
        return self.B[i0] == 1 and self.B[i1] == 1

    def insert(self, item):
        i0, i1 = self._hashes(item)
        self.B[i0] = 1
        self.B[i1] = 1

    def show(self, label=""):
        print(f"{label:<45} B = {self.B}")


bf = TracedBloomFilter(m=8)
bf.show("Initial state:")

# Step 1: insert cnn.com (h0=2, h1=5) -- check it's not already "in" first
item = "http://www.cnn.com/"
print(f"\nBefore inserting {item}: lookup says present = {bf.lookup(item)}")
bf.insert(item)
bf.show(f"After inserting {item}:")

# Step 2: insert helsinki.fi (h0=5, h1=0) -- note h0 collides with cnn's h1,
# but h1=0 is still zero, so it's correctly identified as new
item = "http://www.helsinki.fi/"
print(f"\nBefore inserting {item}: lookup says present = {bf.lookup(item)}")
bf.insert(item)
bf.show(f"After inserting {item}:")

# Step 3: see cnn.com a SECOND time -- both its bits are set, correctly
# recognised as a repeat, filter is unchanged (lookup, not insert)
item = "http://www.cnn.com/"
print(f"\nSeeing {item} again: lookup says present = {bf.lookup(item)} "
      f"(correct -- this really was seen before)")

# Step 4: try yle.fi (h0=0, h1=2) -- NEVER inserted, but both bits happen to
# already be set from cnn.com (bit 2) and helsinki.fi (bit 0) -- a genuine
# false positive purely from hash collisions with OTHER items
item = "http://www.yle.fi/"
print(f"\nFirst time seeing {item}: lookup says present = {bf.lookup(item)} "
      f"(FALSE POSITIVE -- never actually inserted!)")


Initial state:                                B = [0, 0, 0, 0, 0, 0, 0, 0]

Before inserting http://www.cnn.com/: lookup says present = False
After inserting http://www.cnn.com/:          B = [0, 0, 1, 0, 0, 1, 0, 0]

Before inserting http://www.helsinki.fi/: lookup says present = False
After inserting http://www.helsinki.fi/:      B = [1, 0, 1, 0, 0, 1, 0, 0]

Seeing http://www.cnn.com/ again: lookup says present = True (correct -- this really was seen before)

First time seeing http://www.yle.fi/: lookup says present = True (FALSE POSITIVE -- never actually inserted!)


This is the whole mechanism in miniature: **yle.fi** was never inserted,
but its two hash positions (0 and 2) happen to have already been set to 1 by
**cnn.com** (which set bit 2) and **helsinki.fi** (which set bit 0). The filter
has no way to tell "these bits are 1 because of *you*" from "these bits are 1
because of *someone else entirely*" — that ambiguity is the price of the
memory savings, and it's exactly what the false-positive-rate formula below
quantifies.


## 12. Tuning a Bloom filter — what actually controls the false positive rate

Three levers exist, and each comes with a trade-off:

- **Bigger *m* (more bits) lowers the false-positive rate** — but the entire
  point of using a Bloom filter is usually to save memory, so this lever is
  the one you generally don't want to lean on too hard.
- **Too small a *k*** — in the extreme, *k = 1* — suffers from frequent hash
  collisions (the same underlying phenomenon as the birthday paradox), so
  *k = 1* should generally be avoided where possible.
- **Too large a *k*** causes the opposite problem: since every insertion sets
  *k* separate bits to 1, a large *k* fills the bit array with ones very
  quickly, which *also* drives the false-positive rate back up — and it slows
  every lookup down too, since each bit check is effectively a random memory
  access, and there are now more of them per lookup.

The **optimal *k*** therefore depends on both the available memory *m* and the
number of items *n* actually being inserted — there's a genuine sweet spot in
the middle, not a "more is always better" relationship for *k*.


## 13. Deriving the false-positive probability

Walking through the derivation the lecture gives, step by step:

1. The probability that a single hash function sets a *specific* bit *b* to
   one is **1/m** (each hash is assumed uniformly distributed over the *m*
   positions).
2. So the probability that a given bit is *not* set by any one of the *k* hash
   functions for a single insertion is **(1 − 1/m)^k**.
3. After inserting *n* distinct items (each contributing *k* hash evaluations),
   the probability that a specific bit is **still 0** is
   **(1 − 1/m)^(kn)**.
4. A false positive happens when, for some item that was *never* inserted, all
   *k* of its hash positions *happen* to already be set to 1 by other items.
   The probability of that is the probability that a bit *is* set to 1
   — i.e. one minus the "still 0" probability from step 3 — raised to the
   power *k* (since all *k* positions independently need to already be 1):

$$P(\text{false positive}) = \left(1 - \left(1 - \frac{1}{m}\right)^{kn}\right)^k \approx \left(1 - e^{-kn/m}\right)^k$$

The approximation on the right replaces *(1 − 1/m)^(kn)* with *e^(−kn/m)*,
which is accurate for reasonably large *m* — this is the standard
*(1 − 1/m)^m ≈ e^(−1)* approximation from probability theory, extended here.


In [2]:
import math

def false_positive_exact(m, k, n):
    """Exact false-positive probability formula (no e^-x approximation)."""
    prob_bit_still_zero = (1 - 1/m) ** (k * n)
    return (1 - prob_bit_still_zero) ** k

def false_positive_approx(m, k, n):
    """The e^-x approximation the lecture gives, used in practice for its
    simpler closed form and because it's what the optimal-k formula below is
    derived from."""
    return (1 - math.exp(-k * n / m)) ** k

# Sanity check: for a reasonably large m, the two formulas should agree closely
m, k, n = 100_000, 4, 5_000
exact = false_positive_exact(m, k, n)
approx = false_positive_approx(m, k, n)
print(f"m={m}, k={k}, n={n}")
print(f"  exact formula:  {exact:.6f}")
print(f"  e^-x approx:    {approx:.6f}")
print(f"  difference:     {abs(exact - approx):.8f}")


m=100000, k=4, n=5000
  exact formula:  0.001080
  e^-x approx:    0.001080
  difference:     0.00000002


## 14. The optimal number of hash functions

Minimizing the false-positive probability from the formula above over *k*
(for fixed *m* and *n*) gives a clean closed-form answer:

$$k_{\text{optimal}} = \frac{m}{n}\ln(2) \approx 0.693 \, \frac{m}{n}$$

The derivation itself is nontrivial calculus and is left to the literature —
the reference given is Andrei Broder and Michael Mitzenmacher, *Network
Applications of Bloom Filters: A Survey*, Internet Mathematics 1(4): 485–509
(2004). <http://dx.doi.org/10.1080/15427951.2004.10129096>

Notice the intuitive shape of the formula: *k_optimal* scales with the ratio of
available bits to items stored (*m/n*, "bits per item"), not with either
quantity alone. More bits per item allotted → you can afford more hash
functions before the array starts saturating with ones; fewer bits per item →
you need to keep *k* small to avoid filling the array too fast.


## 15. Worked example — sizing a real Bloom filter

**The problem, as posed in the lecture:** the web-caching system needs to
handle **10 million unique URLs**, and there's **12 megabytes** of memory
budgeted for the Bloom filter. Two questions: what's the optimal *k*, and
what's the resulting false-positive rate?


In [3]:
import math

# Given
n = 10_000_000                    # number of unique items to insert
memory_mb = 12
m = memory_mb * 8 * 1024 * 1024   # 12 MB expressed in BITS (8 bits/byte)

print(f"m (bits available)  = {m:,}")
print(f"n (items to insert)  = {n:,}")
print(f"bits per item (m/n)  = {m/n:.3f}")

# Step 1: optimal k
k_exact = (m / n) * math.log(2)
k_optimal = round(k_exact)
print(f"\nOptimal k (exact, unrounded) = {k_exact:.3f}")
print(f"Optimal k (rounded to nearest integer) = {k_optimal}")

# Step 2: false positive rate with that k, for the LAST inserted item
# (i.e. after all n items have gone in, using the approximation formula)
def false_positive_approx(m, k, n):
    return (1 - math.exp(-k * n / m)) ** k

fp_rate = false_positive_approx(m, k_optimal, n)
print(f"\nFalse positive rate with k={k_optimal}: {fp_rate:.4f} = {fp_rate:.1%}")


m (bits available)  = 100,663,296
n (items to insert)  = 10,000,000
bits per item (m/n)  = 10.066

Optimal k (exact, unrounded) = 6.977
Optimal k (rounded to nearest integer) = 7

False positive rate with k=7: 0.0079 = 0.8%


### Reading the result

With 12 MB of memory and 10 million unique URLs, the optimal number of hash
functions comes out to **k ≈ 6.98**, rounding to **k = 7**. Plugging that back
into the false-positive formula gives a rate of roughly **0.8%** for the last
item inserted — meaning about 8 times out of 1,000, a URL genuinely being seen
for the first time will be mistakenly treated as a repeat and cached
prematurely. For a caching application that explicitly said a small false
positive rate was acceptable, that's a very comfortable number.


## 16. The memory savings this actually buys

To see why this matters, it's worth comparing directly against the "traditional
hash table of every URL" design from earlier.

**Given:** average URL length of 25 bytes, worst case where every one of the
10 million URLs is genuinely unique (so nothing can be deduplicated before
storing it).


In [4]:
avg_url_bytes = 25
n = 10_000_000

hash_table_bytes = avg_url_bytes * n
hash_table_mb = hash_table_bytes / (1024 * 1024)

bloom_filter_mb = 12  # from the sizing exercise above

savings_mb = hash_table_mb - bloom_filter_mb

print(f"Traditional hash table of raw URL strings: ~{hash_table_mb:.0f} MB")
print(f"Bloom filter (same 10M items, 0.8% false-positive rate): {bloom_filter_mb} MB")
print(f"Memory saved: ~{savings_mb:.0f} MB")
print(f"Reduction factor: ~{hash_table_mb / bloom_filter_mb:.1f}x smaller")


Traditional hash table of raw URL strings: ~238 MB
Bloom filter (same 10M items, 0.8% false-positive rate): 12 MB
Memory saved: ~226 MB
Reduction factor: ~19.9x smaller


Storing the raw URL strings in an exact hash table costs roughly
**238 MB**; the Bloom filter, accepting a 0.8% false-positive rate, does the
same job in **12 MB** — a savings of **~226 MB**, roughly a **20× reduction**
in memory footprint for this workload. That's the entire economic argument for
using a probabilistic data structure in the first place: a small, controlled,
and precisely quantifiable amount of inaccuracy bought back an enormous amount
of the scarcest resource in the system.


## 17. A general-purpose Bloom filter, for comparison

The traced example above hard-codes the lecture's exact hash outputs to match
the slides precisely. Here's what an actual general-purpose implementation
looks like — using real hash functions rather than a lookup table — so the
mechanism is clear beyond just the one worked trace.


In [5]:
import hashlib

class BloomFilter:
    def __init__(self, m_bits, k_hashes):
        self.m = m_bits
        self.k = k_hashes
        self.bits = [0] * m_bits

    def _positions(self, item):
        # Derive k independent-enough hash positions from k differently-salted
        # SHA-256 digests. Production systems typically use faster
        # non-cryptographic hashes (e.g. MurmurHash) combined via double
        # hashing -- this is simple and clear, not the fastest option.
        item_bytes = str(item).encode()
        for i in range(self.k):
            digest = hashlib.sha256(str(i).encode() + item_bytes).digest()
            yield int.from_bytes(digest, "big") % self.m

    def insert(self, item):
        for pos in self._positions(item):
            self.bits[pos] = 1

    def might_contain(self, item):
        return all(self.bits[pos] == 1 for pos in self._positions(item))


# Reproduce the sizing exercise above, this time actually inserting real data
# and empirically measuring the false-positive rate, rather than relying only
# on the closed-form formula.
import math
import random

n_items = 200_000          # scaled down from 10M for a quick in-notebook run
memory_kb = 240             # scaled down proportionally (12MB * 200k/10M ~ 240KB)
m_bits = memory_kb * 8 * 1024
k_optimal = round((m_bits / n_items) * math.log(2))

print(f"m = {m_bits:,} bits, n = {n_items:,} items, k = {k_optimal}")

bf = BloomFilter(m_bits=m_bits, k_hashes=k_optimal)

random.seed(42)
inserted = [f"https://example.com/page/{i}" for i in range(n_items)]
for url in inserted:
    bf.insert(url)

# Test against a fresh batch of URLs that were NEVER inserted
never_inserted = [f"https://example.com/other/{i}" for i in range(50_000)]
false_positives = sum(1 for url in never_inserted if bf.might_contain(url))
empirical_rate = false_positives / len(never_inserted)

theoretical_rate = (1 - math.exp(-k_optimal * n_items / m_bits)) ** k_optimal

print(f"\nEmpirical false-positive rate (measured):  {empirical_rate:.4%}")
print(f"Theoretical false-positive rate (formula):  {theoretical_rate:.4%}")

# And confirm every genuinely inserted item is still found -- no false negatives, ever
all_found = all(bf.might_contain(url) for url in inserted)
print(f"\nAll {n_items:,} genuinely inserted items still found: {all_found}")


m = 1,966,080 bits, n = 200,000 items, k = 7

Empirical false-positive rate (measured):  0.8700%
Theoretical false-positive rate (formula):  0.8898%

All 200,000 genuinely inserted items still found: True


The empirical measurement and the closed-form formula should land close
to each other — small differences come from the finite sample size of the
"never inserted" test batch and from SHA-256-derived positions not being
*perfectly* uniform, but the formula is a very good practical predictor. And
critically: every genuinely inserted item is still found with 100% certainty —
that guarantee never degrades, no matter how full the filter gets. Only the
false-positive rate on *never-inserted* items gets worse as the filter fills up.


## 18. Bloom filters — what they're good for

- **Memory efficient** for representing sets, by design — that's the entire
  premise, and the worked example above shows a roughly 20× reduction is
  entirely realistic.
- **Very simple to implement**, and very fast to query, particularly with a
  small number of hash functions.
- **Composes naturally with a traditional index.** A common production pattern:
  check the Bloom filter first. If it says "definitely not present," skip the
  traditional index entirely — no disk I/O needed. If it says "maybe present,"
  *then* consult the real index (which may itself live on slow disk). This is
  exactly the pattern lecture 5 already introduced with its Bloom-filter code
  example.
- **Easy to parallelize when building one.** Splitting the data across workers,
  building a separate Bloom filter for each subset, and then combining them
  with a simple **bitwise OR** produces exactly the same filter as if every
  item had been inserted into one filter sequentially — no coordination needed
  during construction beyond the final merge.
- **Many specialized variants exist.** The lecture flags **stable Bloom
  filters** (Deng and Rafiei, SIGMOD 2006) as a notable one, designed for
  streaming data: they probabilistically "forget" older items over time, which
  is exactly the right behaviour for a data stream where the set of interest
  keeps shifting and unbounded memory growth isn't an option.


In [6]:
# A small demonstration of the "OR of parallel Bloom filters" property.

def build_filter(items, m_bits, k_hashes):
    bf = BloomFilter(m_bits=m_bits, k_hashes=k_hashes)
    for item in items:
        bf.insert(item)
    return bf

m_bits, k_hashes = 4096, 4
all_items = [f"item-{i}" for i in range(500)]
half = len(all_items) // 2

# Build one filter sequentially over everything...
bf_sequential = build_filter(all_items, m_bits, k_hashes)

# ...versus building two filters in "parallel" over each half, then OR-ing them
bf_part1 = build_filter(all_items[:half], m_bits, k_hashes)
bf_part2 = build_filter(all_items[half:], m_bits, k_hashes)
bf_merged_bits = [b1 | b2 for b1, b2 in zip(bf_part1.bits, bf_part2.bits)]

print("Sequential filter equals OR-merged parallel filters:",
      bf_sequential.bits == bf_merged_bits)


Sequential filter equals OR-merged parallel filters: True


## 19. Bloom filters — where they fall short

- **No delete operation**, in the basic version. Because a single bit can be
  shared by several items' hash positions, clearing a bit to "delete" one item
  risks silently breaking lookups for other items that also happen to rely on
  that same bit — there's no safe way to undo an insertion without extra
  machinery.
- **k needs tuning to the expected data volume *n*.** Get *n* substantially
  wrong and the filter you built for one workload performs poorly on another —
  this isn't a "set once and forget" parameter.
- **Very low false-positive rates require a large *k***, and a large *k*
  makes the filter slow — every lookup does *k* essentially-random memory
  accesses, so there's a genuine precision-vs-speed trade-off baked in, not
  just a precision-vs-memory one.
- **Large Bloom filters aren't cache-friendly.** Once a filter is bigger than
  the CPU's cache, each of those *k* random bit lookups risks being a genuine
  cache miss, which is far slower than the "very fast" reputation Bloom filters
  otherwise have.
- **User-controllable input can be a problem.** If an adversary can choose
  which items get inserted, they may be able to deliberately engineer hash
  collisions to drive the false-positive rate up on purpose — defending against
  that can require slow, cryptographically secure hash functions, undercutting
  much of the speed advantage.
- **Computing many hash functions can itself be expensive**, particularly at
  high throughput — this is part of why *k* shouldn't be pushed higher than the
  optimal-*k* formula actually calls for.
- **More memory-efficient alternatives that support deletion exist** — the
  lecture points specifically to **Cuckoo filters** (Fan et al., CoNEXT 2014)
  as a notable successor design that solves the "no delete" limitation while
  staying competitive on memory efficiency.


## Further references

- Andrei Broder and Michael Mitzenmacher, *Network Applications of Bloom
  Filters: A Survey*, Internet Mathematics 1(4): 485–509 (2004).
  <http://dx.doi.org/10.1080/15427951.2004.10129096>
- Fan Deng and Davood Rafiei, *Approximately Detecting Duplicates for
  Streaming Data Using Stable Bloom Filters*, SIGMOD Conference 2006: 25–36.
  <http://dx.doi.org/10.1145/1142473.1142477>
- Bin Fan, David G. Andersen, Michael Kaminsky, Michael Mitzenmacher,
  *Cuckoo Filter: Practically Better Than Bloom*, CoNEXT 2014: 75–88.
  <http://dx.doi.org/10.1145/2674005.2674994>


## Summary

This lecture has two halves that connect through a common thread — memory and
consistency are both scarce resources, and every design here is really about
deciding what to spend them on. The NoSQL taxonomy (key-value, document,
extensible record, scalable relational) is four different answers to "which
RDBMS guarantee are we willing to give up, and what do we gain in exchange,"
with BASE/eventual consistency as the most extreme point on that spectrum and
Google Spanner as the system that refuses to make the trade at all — buying its
way out with genuinely exotic hardware (atomic clocks) most organizations
don't have access to. Bloom filters push the same "spend a controlled amount of
imprecision to buy back a scarce resource" idea into an entirely different
domain: not consistency, but RAM itself. A 12 MB Bloom filter standing in for a
238 MB exact hash table, at the cost of a well-understood, precisely tunable
0.8% false-positive rate, is the cleanest illustration in this whole course of
a recurring idea: at cloud scale, giving up a small, *quantified* amount of
correctness is often the only way to make a system affordable to run at all.
